# Bedrock Multi-model Comparison

**Structure:** a shared foundation (config, invocation, helpers) that you run once,
then one self-contained section per challenge. Each challenge only defines its own
prompt, schema, and validator - everything else is reused.

- **Foundation** - run these cells first.
- **Challenge 1 - Structured Output** - extract fields from a support email as JSON.
- **Challenge 2 - Reasoning** - solve a costing problem; validator checks the math.
- **Challenge 3 - Tone Control** - explain cloud computing for two audiences; compared by readability + jargon metrics plus the raw text (no JSON).

For every model the baseline is **prompt-engineered JSON**; **tool calling is optional
and only runs for the single model in `TOOL_MODEL`**, in sections that include a tool-vs-prompt
cell (Challenge 1 does; Challenges 2 and 3 are prompt-only).

## Foundation

### F1. Install & imports

In [5]:
%pip install boto3 pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [6]:
import re
import json
import time

import boto3
import pandas as pd
from botocore.exceptions import ClientError

### F2. Config
Edit `MODELS` to IDs enabled in your region. `TOOL_MODEL` is the one model the tool-vs-prompt cells use. `PRICING` values are **placeholders - verify them**.

In [8]:
REGION = "us-east-1"

# Edit to models enabled in your region.
MODELS = [
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "amazon.nova-lite-v1:0",
    "us.meta.llama3-1-8b-instruct-v1:0",
    "google.gemma-3-4b-it",
]

# The ONE model the tool-vs-prompt cells test with tool calling:
TOOL_MODEL = "amazon.nova-lite-v1:0"

TEMPERATURE = 0.0
MAX_TOKENS = 512


PRICING = {
    "us.anthropic.claude-haiku-4-5-20251001-v1:0": (0.00100, 0.00500),
    "amazon.nova-lite-v1:0":                       (0.00006, 0.00024),
    "us.meta.llama3-1-8b-instruct-v1:0":          (0.00022, 0.00022),
    "google.gemma-3-4b-it":                        (0.00004, 0.00008),
}

### F3. Invocation paths (generic)
These take the prompt/schema as arguments, so every challenge reuses them unchanged.

In [10]:
class ToolUseUnsupported(Exception):
    """Model rejected toolConfig entirely."""


class ToolNotCalled(Exception):
    """Model accepted tools but replied with text instead of calling one."""


def run_prompt(client, model_id, user_text, system_text):
    """Prompt-instructed JSON. Returns (parsed_or_None, meta)."""
    inference = {"maxTokens": MAX_TOKENS, "temperature": TEMPERATURE}

    start = time.time()
    try:
        resp = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": user_text}]}],
            system=[{"text": system_text}],
            inferenceConfig=inference,
        )
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        msg = e.response["Error"]["Message"].lower()
        if code_ == "ValidationException" and "system" in msg:
            resp = client.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": system_text + "\n\n" + user_text}]}],
                inferenceConfig=inference,
            )
        else:
            raise
    latency = time.time() - start

    # Some models put a reasoning block first - grab the text block, not [0].
    blocks = resp["output"]["message"]["content"]
    raw = next((b["text"] for b in blocks if "text" in b), "")
    meta = {"latency": latency, "usage": resp.get("usage", {}),
            "clean": False, "used_fence": False, "raw": raw.strip()}

    parsed = None
    try:
        parsed = json.loads(raw)
        meta["clean"] = True
    except json.JSONDecodeError:
        if "```" in raw:
            meta["used_fence"] = True
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group(0))
            except json.JSONDecodeError:
                parsed = None
    return parsed, meta


def run_tool(client, model_id, user_text, tool_spec):
    """Schema-enforced JSON via tool calling. Raises ToolUseUnsupported / ToolNotCalled."""
    tool_name = tool_spec["toolSpec"]["name"]
    inference = {"maxTokens": MAX_TOKENS, "temperature": TEMPERATURE}
    messages = [{"role": "user", "content": [{"text": user_text}]}]

    def _call(tool_choice):
        return client.converse(
            modelId=model_id,
            messages=messages,
            toolConfig={"tools": [tool_spec], "toolChoice": tool_choice},
            inferenceConfig=inference,
        )

    start = time.time()
    try:
        resp = _call({"tool": {"name": tool_name}})
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        msg = e.response["Error"]["Message"].lower()
        if code_ == "ValidationException":
            if "toolchoice" in msg or "tool choice" in msg:
                resp = _call({"auto": {}})   # e.g. Llama: tools yes, forced choice no
            else:
                raise ToolUseUnsupported(msg) from e
        else:
            raise
    latency = time.time() - start

    meta = {"latency": latency, "usage": resp.get("usage", {})}
    for block in resp["output"]["message"]["content"]:
        if "toolUse" in block:
            raw_input = block["toolUse"]["input"]
            # Reorder keys to match the schema's `properties` declaration order.
            schema_order = list(tool_spec["toolSpec"]["inputSchema"]["json"]["properties"].keys())
            ordered = {k: raw_input[k] for k in schema_order if k in raw_input}
            # Append any extra keys the model returned that aren't in the schema.
            ordered.update({k: v for k, v in raw_input.items() if k not in ordered})
            return ordered, meta
    raise ToolNotCalled()

### F4. Shared helpers
`run_challenge` runs the prompt baseline over all models; `build_scorecard` aggregates; `tool_vs_prompt` handles the optional one-model tool comparison. Each takes a `check_fn` so validation is per-challenge.

In [12]:
def cost(model_id, usage):
    price = PRICING.get(model_id)
    if not price or not usage:
        return None
    in_tok = usage.get("inputTokens", 0)
    out_tok = usage.get("outputTokens", 0)
    return (in_tok / 1000) * price[0] + (out_tok / 1000) * price[1]


def make_record(model_id, task, method, parsed, meta, check_fn):
    keys_ok, values_ok, problems = (
        check_fn(parsed) if parsed is not None else (False, False, ["no JSON parsed"])
    )
    usage = meta.get("usage", {})
    return {
        "model": model_id.split(":")[0],
        "task": task,
        "method": method,
        "valid_json": parsed is not None,
        "clean": bool(meta.get("clean")) if method == "prompt" else None,
        "values_ok": values_ok,   # structured: fields valid | reasoning: math + rec correct
        "latency_s": round(meta.get("latency", 0.0), 3),
        "in_tok": usage.get("inputTokens", 0),
        "out_tok": usage.get("outputTokens", 0),
        "cost_usd": cost(model_id, usage),
        "problems": "; ".join(problems),
        "parsed": parsed,
    }


def run_challenge(user_text, system_text, tool_spec, check_fn, label, show=True):
    """Prompt-engineered JSON baseline across every model. Returns a list of records."""
    records = []
    for model_id in MODELS:
        print("=== " + model_id + " ===")
        try:
            parsed, meta = run_prompt(client, model_id, user_text, system_text)
        except ClientError as e:
            err = e.response["Error"]
            print("  unavailable - " + err["Code"] + ": " + err["Message"] + "\n")
            continue
        records.append(make_record(model_id, label, "prompt", parsed, meta, check_fn))
        if show:
            _, values_ok, _ = check_fn(parsed) if parsed is not None else (False, False, None)
            print(f"  ({meta['latency']:.2f}s, clean={meta['clean']}, ok={values_ok})")
            print(json.dumps(parsed, indent=2, ensure_ascii=False) if parsed is not None else meta["raw"])
            print()
    print("rows: " + str(len(records)))
    return records


def build_scorecard(df):
    def rate(s):
        s = s.dropna()
        return f"{int(s.sum())}/{len(s)}" if len(s) else "-"
    sc = (
        df.groupby(["model", "method"])
          .agg(json=("valid_json", rate),
               clean=("clean", rate),
               ok=("values_ok", rate),
               avg_latency_s=("latency_s", "mean"),
               total_cost_usd=("cost_usd", lambda s: s.sum(min_count=1)))
          .reset_index()
    )
    sc["avg_latency_s"] = sc["avg_latency_s"].round(2)
    sc["total_cost_usd"] = sc["total_cost_usd"].round(5)
    return sc


def tool_vs_prompt(model_id, user_text, system_text, tool_spec, check_fn):
    """Run one model both ways, print both JSON outputs, compare latency."""
    print("TOOL vs PROMPT for: " + model_id + "\n")

    p_parsed, p_meta = run_prompt(client, model_id, user_text, system_text)

    tool_ok = True
    try:
        t_parsed, t_meta = run_tool(client, model_id, user_text, tool_spec)
    except ToolUseUnsupported:
        tool_ok = False
        print("This model does not support tool calling - prompt output only.\n")
    except ToolNotCalled:
        tool_ok = False
        print("Model offered tools but returned text instead.\n")

    def show(tag, parsed, meta):
        _, values_ok, _ = check_fn(parsed) if parsed is not None else (False, False, None)
        c = cost(model_id, meta.get("usage", {}))
        cstr = f"${c:.5f}" if c is not None else "n/a"
        print(f"[{tag}] {meta['latency']:.2f}s  ok={values_ok}  cost={cstr}")
        print(json.dumps(parsed, indent=2, ensure_ascii=False) if parsed is not None else meta.get("raw", ""))
        print()

    show("prompt", p_parsed, p_meta)
    if tool_ok:
        show("tool", t_parsed, t_meta)
        diff = t_meta["latency"] - p_meta["latency"]
        faster = "tool" if diff < 0 else "prompt"
        print(f"Latency delta: {abs(diff):.2f}s  ({faster} faster)")


DISPLAY_COLS = ["model", "task", "method", "valid_json", "clean", "values_ok",
                "latency_s", "in_tok", "out_tok", "cost_usd", "problems", "parsed"]

### F5. Bedrock client

In [14]:
client = boto3.client("bedrock-runtime", region_name=REGION)
# Optional credential check:
# boto3.client("sts", region_name=REGION).get_caller_identity()

## Challenge 1 - Structured Output

Give each model a customer support email and have it extract `name`, `issue`, `sentiment`,
`urgency` as a JSON object, JSON only. `values_ok` here = all keys present and sentiment/urgency
within their allowed sets.

### C1.1 Task definition

In [17]:
EMAIL = """Subject: Still no refund - running out of patience

My name is Priya Nair. I returned the wireless headphones three weeks ago and
was promised a refund in 5 business days. I've emailed twice with no reply.
Resolve this today or I'm disputing the charge with my bank.
Priya"""


def c1_request(email):
    return (
        "Extract the customer's name, their issue, the sentiment "
        "(positive / neutral / negative), and the urgency (low / medium / high) "
        "from this customer support email:\n\n"
        f'"""\n{email}\n"""'
    )

C1_SYSTEM = (
    "You are a precise information-extraction engine. Respond with a SINGLE "
    "valid JSON object and nothing else - no explanations, no markdown, no "
    "code fences."
)
C1_SUFFIX = (
    '\n\nReturn a single JSON object with exactly these keys: '
    '"name", "issue", "sentiment", "urgency". Output JSON only.'
)

C1_KEYS = {"name", "issue", "sentiment", "urgency"}
ALLOWED_SENTIMENT = {"positive", "neutral", "negative"}
ALLOWED_URGENCY = {"low", "medium", "high"}

C1_TOOL = {
    "toolSpec": {
        "name": "record_ticket_fields",
        "description": "Record the key fields extracted from a customer support email.",
        "inputSchema": {"json": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Customer's full name, or empty if none."},
                "issue": {"type": "string", "description": "Short description of the problem."},
                "sentiment": {"type": "string", "enum": sorted(ALLOWED_SENTIMENT)},
                "urgency": {"type": "string", "enum": sorted(ALLOWED_URGENCY)},
            },
            "required": ["name", "issue", "sentiment", "urgency"],
        }},
    }
}


def check_structured(parsed):
    """Return (keys_ok, values_ok, problems[])."""
    if not isinstance(parsed, dict):
        return False, False, ["not a JSON object"]

    missing = C1_KEYS - set(parsed.keys())
    keys_ok = not missing
    problems = [f"missing {k}" for k in sorted(missing)]

    for field, allowed in (("sentiment", ALLOWED_SENTIMENT), ("urgency", ALLOWED_URGENCY)):
        if field in parsed and str(parsed[field]).strip().lower() not in allowed:
            problems.append(f"{field}={parsed[field]!r} not in {sorted(allowed)}")
    for field in ("name", "issue"):
        if field in parsed and not (isinstance(parsed[field], str) and parsed[field].strip()):
            problems.append(f"{field} empty/non-string")

    values_ok = keys_ok and not [p for p in problems if not p.startswith("missing")]
    return keys_ok, values_ok, problems


c1_user = c1_request(EMAIL) + C1_SUFFIX

### C1.2 Run - prompt baseline for every model

In [19]:
c1_records = run_challenge(c1_user, C1_SYSTEM, C1_TOOL, check_structured, "structured_output")

=== us.anthropic.claude-haiku-4-5-20251001-v1:0 ===
  (6.82s, clean=False, ok=True)
{
  "name": "Priya Nair",
  "issue": "Refund not received for returned wireless headphones after three weeks; no response to follow-up emails",
  "sentiment": "negative",
  "urgency": "high"
}

=== amazon.nova-lite-v1:0 ===
  (0.65s, clean=False, ok=True)
{
  "name": "Priya Nair",
  "issue": "Still no refund after returning wireless headphones three weeks ago.",
  "sentiment": "negative",
  "urgency": "high"
}

=== us.meta.llama3-1-8b-instruct-v1:0 ===
  (0.57s, clean=True, ok=True)
{
  "name": "Priya Nair",
  "issue": "Refund for returned wireless headphones",
  "sentiment": "negative",
  "urgency": "high"
}

=== google.gemma-3-4b-it ===
  (0.55s, clean=False, ok=True)
{
  "name": "Priya Nair",
  "issue": "Refund for returned wireless headphones not received",
  "sentiment": "negative",
  "urgency": "high"
}

rows: 4


### C1.3 Raw results

In [21]:
c1_df = pd.DataFrame(c1_records)
c1_df[DISPLAY_COLS]

,model,task,method,valid_json,clean,values_ok,latency_s,in_tok,out_tok,cost_usd,problems,parsed
0,us.anthropic.claude-haiku-4-5-20251001-v1,structured_output,prompt,True,False,True,6.816,183,69,0.000528,,"{'name': 'Priya Nair', 'issue': 'Refund not re..."
1,amazon.nova-lite-v1,structured_output,prompt,True,False,True,0.647,159,47,0.000021,,"{'name': 'Priya Nair', 'issue': 'Still no refu..."
2,us.meta.llama3-1-8b-instruct-v1,structured_output,prompt,True,True,True,0.572,173,42,0.000047,,"{'name': 'Priya Nair', 'issue': 'Refund for re..."
3,google.gemma-3-4b-it,structured_output,prompt,True,False,True,0.552,173,50,0.000011,,"{'name': 'Priya Nair', 'issue': 'Refund for re..."


### C1.4 Scorecard

In [23]:
build_scorecard(c1_df)

,model,method,json,clean,ok,avg_latency_s,total_cost_usd
0,amazon.nova-lite-v1,prompt,1/1,0/1,1/1,0.65,0.00002
1,google.gemma-3-4b-it,prompt,1/1,0/1,1/1,0.55,0.00001
2,us.anthropic.claude-haiku-4-5-20251001-v1,prompt,1/1,0/1,1/1,6.82,0.00053
3,us.meta.llama3-1-8b-instruct-v1,prompt,1/1,1/1,1/1,0.57,0.00005


### C1.5 Tool calling vs prompt (one model)

In [25]:
tool_vs_prompt(TOOL_MODEL, c1_user, C1_SYSTEM, C1_TOOL, check_structured)

TOOL vs PROMPT for: amazon.nova-lite-v1:0

[prompt] 0.72s  ok=True  cost=$0.00002
{
  "name": "Priya Nair",
  "issue": "Still no refund after returning wireless headphones three weeks ago.",
  "sentiment": "negative",
  "urgency": "high"
}

[tool] 0.88s  ok=True  cost=$0.00005
{
  "name": "Priya Nair",
  "issue": "Still no refund after returning wireless headphones",
  "sentiment": "negative",
  "urgency": "high"
}

Latency delta: 0.16s  (prompt faster)


## Challenge 2 - Reasoning

Give each model a short costing problem, and have it compute both options, pick the cheaper,
and recommend. Unlike Challenge 1, the validator checks the **arithmetic against the known
answer**, not just the JSON shape.

**Problem:** 1,200 orders next month. Courier A = \$500 flat + \$2.50/order. Courier B = \$6.00/order, no fee.

**Correct answer:** A = 500 + 2.50x1200 = **\$3,500**; B = 6.00x1200 = **\$7,200**; A is cheaper by **\$3,700**.

`values_ok` here = the model's numbers match those (within a small tolerance) **and** it picks Courier A.

### C2.1 Task definition + ground truth

In [28]:
BUSINESS_PROBLEM = (
    "Your team must choose a shipping courier for next month, when you expect "
    "1,200 orders.\n\n"
    "- Courier A: a flat $500 monthly fee plus $2.50 per order.\n"
    "- Courier B: $6.00 per order, with no monthly fee.\n\n"
    "Work out the total cost of each courier for next month, decide which is "
    "cheaper and by how much, then give a one-line recommendation."
)

C2_SYSTEM = (
    "You are a precise financial analyst. Compute carefully, then respond with a "
    "SINGLE valid JSON object and nothing else - no explanations, no markdown, "
    "no code fences."
)
C2_SUFFIX = (
    '\n\nReturn a single JSON object with exactly these keys: '
    '"courier_a_cost", "courier_b_cost", "cheaper_option", "savings", '
    '"recommendation". Costs and savings are numbers in dollars; "cheaper_option" '
    'is "Courier A" or "Courier B". Output JSON only.'
)

# Ground truth for validation.
C2_KEYS = {"courier_a_cost", "courier_b_cost", "cheaper_option", "savings", "recommendation"}
C2_TRUTH_NUM = {"courier_a_cost": 3500.0, "courier_b_cost": 7200.0, "savings": 3700.0}
C2_TRUTH_OPTION = "Courier A"
C2_TOL = 0.5   # dollars


def check_reasoning(parsed):
    """Return (keys_ok, values_ok, problems[]) - values_ok means the MATH is right."""
    if not isinstance(parsed, dict):
        return False, False, ["not a JSON object"]

    missing = C2_KEYS - set(parsed.keys())
    keys_ok = not missing
    problems = [f"missing {k}" for k in sorted(missing)]

    for field, truth in C2_TRUTH_NUM.items():
        if field in parsed:
            try:
                if abs(float(parsed[field]) - truth) > C2_TOL:
                    problems.append(f"{field}={parsed[field]} (expected {truth})")
            except (TypeError, ValueError):
                problems.append(f"{field} not numeric: {parsed[field]!r}")

    if "cheaper_option" in parsed:
        co = parsed["cheaper_option"]
        if str(co).strip().lower() != C2_TRUTH_OPTION.lower():
            problems.append(f"cheaper_option={co!r} (expected {C2_TRUTH_OPTION})")

    values_ok = keys_ok and not [p for p in problems if not p.startswith("missing")]
    return keys_ok, values_ok, problems


c2_user = BUSINESS_PROBLEM + C2_SUFFIX

### C2.2 Run - prompt baseline for every model

In [30]:
c2_records = run_challenge(c2_user, C2_SYSTEM, None, check_reasoning, "reasoning")

=== us.anthropic.claude-haiku-4-5-20251001-v1:0 ===
  (13.71s, clean=False, ok=True)
{
  "courier_a_cost": 3500,
  "courier_b_cost": 7200,
  "cheaper_option": "Courier A",
  "savings": 3700,
  "recommendation": "Choose Courier A to save $3,700 compared to Courier B for 1,200 orders next month."
}

=== amazon.nova-lite-v1:0 ===
  (0.81s, clean=False, ok=True)
{
  "courier_a_cost": 3500,
  "courier_b_cost": 7200,
  "cheaper_option": "Courier A",
  "savings": 3700,
  "recommendation": "Choose Courier A"
}

=== us.meta.llama3-1-8b-instruct-v1:0 ===
  (0.54s, clean=True, ok=False)
{
  "courier_a_cost": 2500,
  "courier_b_cost": 7200,
  "cheaper_option": "Courier A",
  "savings": 4700,
  "recommendation": "Choose Courier A"
}

=== google.gemma-3-4b-it ===
  (0.68s, clean=False, ok=False)
{
  "courier_a_cost": 3100.0,
  "courier_b_cost": 7200.0,
  "cheaper_option": "Courier A",
  "savings": 4100.0,
  "recommendation": "Choose Courier A for significant cost savings."
}

rows: 4


### C2.3 Raw results
`values_ok` = arithmetic correct and Courier A chosen; `problems` shows any wrong number.

In [68]:
pd.set_option('display.max_colwidth', None)

In [70]:
c2_df = pd.DataFrame(c2_records)
c2_df[DISPLAY_COLS]

,model,task,method,valid_json,clean,values_ok,latency_s,in_tok,out_tok,cost_usd,problems,parsed
0,us.anthropic.claude-haiku-4-5-20251001-v1,reasoning,prompt,True,False,True,13.713,201,90,0.000651,,"{'courier_a_cost': 3500, 'courier_b_cost': 7200, 'cheaper_option': 'Courier A', 'savings': 3700, 'recommendation': 'Choose Courier A to save $3,700 compared to Courier B for 1,200 orders next month.'}"
1,amazon.nova-lite-v1,reasoning,prompt,True,False,True,0.814,193,70,0.000028,,"{'courier_a_cost': 3500, 'courier_b_cost': 7200, 'cheaper_option': 'Courier A', 'savings': 3700, 'recommendation': 'Choose Courier A'}"
2,us.meta.llama3-1-8b-instruct-v1,reasoning,prompt,True,True,False,0.535,191,55,0.000054,courier_a_cost=2500 (expected 3500.0); savings=4700 (expected 3700.0),"{'courier_a_cost': 2500, 'courier_b_cost': 7200, 'cheaper_option': 'Courier A', 'savings': 4700, 'recommendation': 'Choose Courier A'}"
3,google.gemma-3-4b-it,reasoning,prompt,True,False,False,0.683,202,88,0.000015,courier_a_cost=3100.0 (expected 3500.0); savings=4100.0 (expected 3700.0),"{'courier_a_cost': 3100.0, 'courier_b_cost': 7200.0, 'cheaper_option': 'Courier A', 'savings': 4100.0, 'recommendation': 'Choose Courier A for significant cost savings.'}"


### C2.4 Scorecard

In [34]:
build_scorecard(c2_df)

,model,method,json,clean,ok,avg_latency_s,total_cost_usd
0,amazon.nova-lite-v1,prompt,1/1,0/1,1/1,0.81,0.00003
1,google.gemma-3-4b-it,prompt,1/1,0/1,0/1,0.68,0.00002
2,us.anthropic.claude-haiku-4-5-20251001-v1,prompt,1/1,0/1,1/1,13.71,0.00065
3,us.meta.llama3-1-8b-instruct-v1,prompt,1/1,1/1,0/1,0.54,0.00005


## Challenge 3 - Tone Control

Ask each model to explain cloud computing twice - once for a 10-year-old, once for a senior
engineer - and see whether it actually adapts register. There is no "correct" answer here, so
this section is shaped differently from 1 and 2:

- **No JSON.** Output is plain prose (`run_text` below, not `run_prompt`).
- **Proxy metrics** per version: reading-grade level, Flesch reading ease, technical-jargon count, word count.
- **The signal is the delta.** A model that controls tone produces a *higher* grade level and *more*
  jargon for the engineer than for the child. Small deltas = weak tone control.
- **Read the text.** Metrics are proxies; the printed outputs are the real evidence.

### C3.1 Setup (installs `textstat` for readability metrics)

In [37]:
%pip install textstat --quiet

Note: you may need to restart the kernel to use updated packages.


In [38]:
import textstat

C3_SYSTEM = (
    "You are a helpful explainer. Reply with a plain-prose explanation only - "
    "no lists, no headings, no code, no JSON."
)

AUDIENCES = {
    "10-year-old": "Explain what cloud computing is to a curious 10-year-old, in a short paragraph.",
    "senior engineer": "Explain what cloud computing is to a senior software engineer, in a short paragraph.",
}

# Technical terms an engineer explanation is likelier to use than a child one.
JARGON = [
    "server", "servers", "virtualization", "virtual", "infrastructure", "scalability",
    "scalable", "scale", "latency", "api", "apis", "provisioning", "provision",
    "distributed", "redundancy", "load balancing", "elastic", "on-demand", "data center",
    "datacenter", "bandwidth", "compute", "storage", "deployment", "deploy", "throughput",
    "container", "orchestration", "hypervisor", "iaas", "paas", "saas", "node", "cluster",
]


def run_text(client, model_id, user_text, system_text):
    """Plain-text (non-JSON) invocation. Returns (text, meta)."""
    inference = {"maxTokens": MAX_TOKENS, "temperature": TEMPERATURE}
    start = time.time()
    try:
        resp = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": user_text}]}],
            system=[{"text": system_text}],
            inferenceConfig=inference,
        )
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        msg = e.response["Error"]["Message"].lower()
        if code_ == "ValidationException" and "system" in msg:
            resp = client.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": system_text + "\n\n" + user_text}]}],
                inferenceConfig=inference,
            )
        else:
            raise
    latency = time.time() - start
    blocks = resp["output"]["message"]["content"]
    text = " ".join(b["text"] for b in blocks if "text" in b).strip()
    return text, {"latency": latency, "usage": resp.get("usage", {})}


def jargon_count(text):
    t = text.lower()
    return sum(len(re.findall(r"\b" + re.escape(term) + r"\b", t)) for term in JARGON)


def metrics(text):
    return {
        "words": textstat.lexicon_count(text, removepunct=True),
        "grade": round(textstat.flesch_kincaid_grade(text), 1),
        "reading_ease": round(textstat.flesch_reading_ease(text), 1),
        "jargon": jargon_count(text),
    }

### C3.2 Run - both audiences for every model

In [40]:
c3_records = []

for model_id in MODELS:
    print("=== " + model_id + " ===")
    for aud, prompt in AUDIENCES.items():
        try:
            text, meta = run_text(client, model_id, prompt, C3_SYSTEM)
        except ClientError as e:
            err = e.response["Error"]
            print("  unavailable - " + err["Code"] + ": " + err["Message"])
            break
        m = metrics(text)
        grade = m["grade"]; ease = m["reading_ease"]; jarg = m["jargon"]; words = m["words"]
        usage = meta["usage"]
        c3_records.append({
            "model": model_id.split(":")[0],
            "audience": aud,
            "grade": grade,
            "reading_ease": ease,
            "jargon": jarg,
            "words": words,
            "latency_s": round(meta["latency"], 3),
            "in_tok": usage.get("inputTokens", 0),
            "out_tok": usage.get("outputTokens", 0),
            "cost_usd": cost(model_id, usage),
            "text": text,
        })
        print(f"  [{aud:<15}] grade={grade}  ease={ease}  jargon={jarg}  words={words}")
    print()

print("rows: " + str(len(c3_records)))

=== us.anthropic.claude-haiku-4-5-20251001-v1:0 ===
  [10-year-old    ] grade=11.2  ease=62.2  jargon=1  words=134
  [senior engineer] grade=25.4  ease=-27.5  jargon=9  words=100

=== amazon.nova-lite-v1:0 ===
  [10-year-old    ] grade=8.4  ease=72.8  jargon=0  words=106
  [senior engineer] grade=19.7  ease=1.8  jargon=6  words=107

=== us.meta.llama3-1-8b-instruct-v1:0 ===
  [10-year-old    ] grade=9.6  ease=64.3  jargon=0  words=129
  [senior engineer] grade=17.4  ease=24.5  jargon=7  words=91

=== google.gemma-3-4b-it ===
  [10-year-old    ] grade=10.7  ease=57.8  jargon=1  words=89
  [senior engineer] grade=14.8  ease=32.6  jargon=4  words=73

rows: 8


### C3.3 Raw metrics

In [42]:
c3_df = pd.DataFrame(c3_records)
c3_df[["model", "audience", "grade", "reading_ease", "jargon", "words", "latency_s", "cost_usd"]]

,model,audience,grade,reading_ease,jargon,words,latency_s,cost_usd
0,us.anthropic.claude-haiku-4-5-20251001-v1,10-year-old,11.2,62.2,1,134,9.671,0.000847
1,us.anthropic.claude-haiku-4-5-20251001-v1,senior engineer,25.4,-27.5,9,100,10.231,0.000798
2,amazon.nova-lite-v1,10-year-old,8.4,72.8,0,106,1.446,0.000034
3,amazon.nova-lite-v1,senior engineer,19.7,1.8,6,107,1.324,0.000035
4,us.meta.llama3-1-8b-instruct-v1,10-year-old,9.6,64.3,0,129,1.119,0.000048
5,us.meta.llama3-1-8b-instruct-v1,senior engineer,17.4,24.5,7,91,0.871,0.000038
6,google.gemma-3-4b-it,10-year-old,10.7,57.8,1,89,1.009,0.000011
7,google.gemma-3-4b-it,senior engineer,14.8,32.6,4,73,0.908,0.000009


### C3.4 Tone-control scorecard
One row per model: the reading grade level of the **kid version** vs the **engineer version**, the
gap between them, and a plain verdict. A big gap (engineer version much harder to read than the kid
version) means the model adapted its tone well. Jargon and other detail live in the raw table above.

In [44]:
def tone_scorecard(df):
    rows = []
    for model, g in df.groupby("model"):
        gg = g.set_index("audience")
        if {"10-year-old", "senior engineer"} <= set(gg.index):
            kid = gg.loc["10-year-old"]["grade"]
            eng = gg.loc["senior engineer"]["grade"]
            gap = eng - kid
            if gap >= 5:
                verdict = "Strong - clearly adapts"
            elif gap >= 2.5:
                verdict = "Moderate"
            else:
                verdict = "Weak - barely adapts"
            rows.append({
                "model": model,
                "kid_level": round(kid),
                "engineer_level": round(eng),
                "gap": round(gap, 1),
                "verdict": verdict,
            })
    return (pd.DataFrame(rows)
              .sort_values("gap", ascending=False)
              .reset_index(drop=True))

tone_scorecard(c3_df)

,model,kid_level,engineer_level,gap,verdict
0,us.anthropic.claude-haiku-4-5-20251001-v1,11,25,14.2,Strong - clearly adapts
1,amazon.nova-lite-v1,8,20,11.3,Strong - clearly adapts
2,us.meta.llama3-1-8b-instruct-v1,10,17,7.8,Strong - clearly adapts
3,google.gemma-3-4b-it,11,15,4.1,Moderate


### C3.5 Read the actual explanations
The metrics are proxies - this is the real evidence of tone control.

In [46]:
for r in c3_records:
    model = r["model"]; aud = r["audience"]; grade = r["grade"]; jarg = r["jargon"]
    print("=" * 72)
    print(f"{model}  |  {aud}   (grade {grade}, jargon {jarg})")
    print("=" * 72)
    print(r["text"])
    print()

us.anthropic.claude-haiku-4-5-20251001-v1  |  10-year-old   (grade 11.2, jargon 1)
Cloud computing is like having a magical computer that lives somewhere far away instead of on your desk. Instead of storing all your games, photos, and homework on your own computer, you can save them on this faraway computer (called a server) that's connected to the internet. Whenever you want to use your stuff, you just ask the internet to get it for you, and it appears on your screen. It's kind of like having a really smart friend who keeps all your things organized in their house, and whenever you need something, they bring it to you right away. The cool part is that you can use your files from any computer or tablet, and you don't have to worry about your computer running out of space because the faraway computer is huge!

us.anthropic.claude-haiku-4-5-20251001-v1  |  senior engineer   (grade 25.4, jargon 9)
Cloud computing is the delivery of computing resources—including servers, storage, databases